In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [12]:
def get_values(symbol):
    t = time.time()
#     print(t)
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(x.year, x.month, 1, tzinfo=timezone)    
    utc_to = datetime(x.year, x.month, x.day+1, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_H6, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)[-50:]
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['tt'] = rates_frame['time']
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
#     rates_frame['rsi'] = pta.rsi(rates_frame['close'], length = 14)
    
#     rates_frame['sma'] = rates_frame['close'].rolling(window=300).mean()
#     rates_frame['smaH'] = rates_frame['high'].rolling(window=30).mean()
#     rates_frame['smaL']= rates_frame['low'].rolling(window=20).mean()
#     rates_frame = rates_frame.drop(['high', 'low'], axis=1)
#     rates_frame = rates_frame
    rates_frame['rsi'] = get_rsi(rates_frame['close'], 14)

#     print(time.time()-t)
    return rates_frame

In [13]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [14]:
symbol = "EURUSD"
a = get_values(symbol)
a

,open,high,low,close,tt,rsi
time,,,,,,
2021-08-06 00:00:00,1.18311,1.18344,1.18188,1.18202,1628208000,NaN
2021-08-06 06:00:00,1.18201,1.18287,1.18060,1.18098,1628229600,0.000000
2021-08-06 12:00:00,1.18099,1.18115,1.17530,1.17592,1628251200,0.000000
2021-08-06 18:00:00,1.17591,1.17641,1.17522,1.17572,1628272800,0.000000
2021-08-09 00:00:00,1.17602,1.17641,1.17403,1.17581,1628467200,1.644931
2021-08-09 06:00:00,1.17580,1.17646,1.17481,1.17528,1628488800,1.489542
2021-08-09 12:00:00,1.17527,1.17670,1.17416,1.17423,1628510400,1.239691
2021-08-09 18:00:00,1.17423,1.17515,1.17329,1.17331,1628532000,1.070290
2021-08-10 00:00:00,1.17354,1.17374,1.17300,1.17360,1628553600,5.455902


In [31]:
# less negatives more positives

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000

lenn = len(a)

for i in range(5, len(a)):
    if check == 0 and a.iloc[i].name.hour == 6:

        buy_price = a.iloc[i].close
        print("#"*20)
        print(a.iloc[i].name)
        print("*"*20)
        check = 1  
        up = 0
        k = 0.0

    elif check == 1:
        sell_price = a.iloc[i].high
        pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
        print(f"{pp}---{a.iloc[i].close}--{a.iloc[i].name}")
        profit.append(pp)
        check = 0
        sell_price = a.iloc[i].low
        pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
        print(f"{pp}---{a.iloc[i].close}--{a.iloc[i].name}")
        profit.append(pp)
        check = 0

####################
2021-08-09 06:00:00
********************
2.84---1.17423--2021-08-09 12:00:00
2.24---1.17423--2021-08-09 12:00:00
####################
2021-08-10 06:00:00
********************
-0.1---1.17171--2021-08-10 12:00:00
4.46---1.17171--2021-08-10 12:00:00
####################
2021-08-11 06:00:00
********************
7.38---1.1737--2021-08-11 12:00:00
0.8---1.1737--2021-08-11 12:00:00
####################
2021-08-12 06:00:00
********************
0.1---1.17278--2021-08-12 12:00:00
4.66---1.17278--2021-08-12 12:00:00
####################
2021-08-13 06:00:00
********************
11.08---1.17923--2021-08-13 12:00:00
0.38---1.17923--2021-08-13 12:00:00
####################
2021-08-16 06:00:00
********************
0.56---1.1787--2021-08-16 12:00:00
4.12---1.1787--2021-08-16 12:00:00
####################
2021-08-17 06:00:00
********************
0.08---1.17168--2021-08-17 12:00:00
12.56---1.17168--2021-08-17 12:00:00
####################
2021-08-18 06:00:00
********************
1.2-

In [32]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

Total negative sm -->-0.1
Total negative -->1
Total positive sm -->70.04
Total positive -->21
Length 22


In [ ]:
#         if a.iloc[i-3].rsi < a.iloc[i-2].rsi and a.iloc[i-1].rsi < a.iloc[i-2].rsi \
#             and a.iloc[i-1].rsi < a.iloc[i].rsi and a.iloc[i].rsi < a.iloc[i-2].rsi \
#             and a.iloc[i].close > a.iloc[i].open \
#             and (a.iloc[i].rsi - a.iloc[i-1].rsi) >= 2.0 \
#              and check == 0:
#             and (a.iloc[i-2].rsi - a.iloc[i-3].rsi) >= 2.0

In [24]:
#less negatives more positives

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000

for i in range(5, len(a)):
    if a.iloc[i-1].rsi < a.iloc[i].rsi and (a.iloc[i].rsi - a.iloc[i-1].rsi) >= 2.0 \
        and a.iloc[i-2].rsi > a.iloc[i-1].rsi \
        and check == 0:

        buy_price = a.iloc[i+1].open
        print("#"*20)
        print(a.iloc[i+1].name)
        print("*"*20)
        check = 1  
        up = 0
        dp = 0
        k = 0.0

    elif check == 1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
        print(f"{pp}---{round(a.iloc[i].rsi, 2)}---{a.iloc[i].close}--{a.iloc[i].name}")

#         if pp >= 0.02:
#             profit.append(pp)
#             check = 0
        if pp > 0.0:
            up = up+1
            if up > 1: 
                profit.append(pp)
                check = 0
            dp = 1
#         elif pp < 0.0:
#             dp = dp+1
#             if dp > 1: 
#                 profit.append(pp)
#                 check = 0
#             up = 0

#             elif pp < -2.0:
#                 profit.append(pp)
#                 check = 0

        elif pp > 0.0:
            up = up+1
            if up == 1:
                k = pp
            if up > 1:
                if pp > k:
                    print("pass")
                else:
                    profit.append(pp)
                    check = 0
            k = pp
                #In live trade if loss still goes on to increase then close the trade before hand

####################
2021-08-10 06:00:00
********************
1.22---4.97---1.173--2021-08-10 06:00:00
3.8---4.11---1.17171--2021-08-10 12:00:00
####################
2021-08-11 06:00:00
********************
2.5---14.08---1.17104--2021-08-11 06:00:00
-2.82---35.53---1.1737--2021-08-11 12:00:00
-2.14---34.35---1.17336--2021-08-11 18:00:00
-2.9---36.88---1.17374--2021-08-12 00:00:00
-4.46---41.82---1.17452--2021-08-12 06:00:00
-0.98---35.2---1.17278--2021-08-12 12:00:00
-0.9---35.06---1.1727400000000001--2021-08-12 18:00:00
-3.24---42.17---1.17391--2021-08-13 00:00:00
-3.56---43.09---1.17407--2021-08-13 06:00:00
-13.88---63.32---1.17923--2021-08-13 12:00:00
-13.76---63.04---1.17917--2021-08-13 18:00:00
-14.86---64.59---1.17972--2021-08-16 00:00:00
-12.86---59.69---1.17872--2021-08-16 06:00:00
-12.82---59.59---1.1787--2021-08-16 12:00:00
-9.94---52.9---1.17726--2021-08-16 18:00:00
-9.84---52.68---1.17721--2021-08-17 00:00:00
-10.72---54.49---1.17765--2021-08-17 06:00:00
1.22---35.0---1.171

In [25]:
n = 0
p = 0
tn = 0
tp = 0
print(sum(profit))
for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

10.18
Total negative sm -->0
Total negative -->0
Total positive sm -->10.18
Total positive -->5
Length 5
